# 📚 Graph RAG 완전 해설 노트북
## `build_cell_graph` & `graph_search` — 초보자를 위한 A to Z 가이드

---

## 🗺️ 이 노트북에서 배울 것

| 단계 | 내용 |
|------|------|
| 1 | Graph RAG가 무엇인지 개념 이해 |
| 2 | 필요한 라이브러리 import |
| 3 | 예제 데이터(셀 목록) 직접 만들기 |
| 4 | `build_cell_graph` 함수 — 줄별 완전 해설 |
| 5 | 그래프 시각화로 눈으로 확인하기 |
| 6 | `graph_search` 함수 — 줄별 완전 해설 |
| 7 | 전체 파이프라인 실행 |

---

## 💡 핵심 개념 먼저 이해하기

### RAG(Retrieval-Augmented Generation)란?
LLM(대형 언어 모델)이 답변을 생성할 때, 외부 문서를 **검색(Retrieve)**해서 활용하는 방식입니다.

```
일반 RAG:  질문 → 벡터 유사도 검색 → 관련 문서 → LLM 답변
Graph RAG: 질문 → 벡터 검색 + 그래프 탐색 → 관련 문서 → LLM 답변
                              ↑
                  "연결된 셀도 함께 가져옴!"
```

### 왜 Graph RAG가 필요한가?
Jupyter 노트북에서 코드 셀은 **서로 연결**되어 있습니다.
- Cell 1에서 `model = ChatOpenAI(...)` 정의
- Cell 5에서 `model.invoke(...)` 사용

→ Cell 5만 검색해서는 `model`이 뭔지 모릅니다!
→ **그래프로 연결 관계를 저장**해두면 Cell 5를 찾을 때 Cell 1도 함께 가져올 수 있습니다.


---
## STEP 0 ─ 라이브러리 Import

각 라이브러리가 왜 필요한지 설명합니다.

In [ ]:
# ── 기본 라이브러리 ────────────────────────────────────────────────────────
import re                        # 정규표현식: 코드에서 변수명을 추출할 때 사용
from pprint import pprint        # 딕셔너리/리스트를 보기 좋게 출력

# ── 그래프 라이브러리 ─────────────────────────────────────────────────────
import networkx as nx            # 그래프 자료구조 (노드 + 엣지)
import matplotlib.pyplot as plt  # 그래프 시각화
import matplotlib.font_manager as fm

# ── LangChain 관련 ────────────────────────────────────────────────────────
from langchain_core.documents import Document  # LangChain의 문서 객체
                                        # Document(page_content=..., metadata={...})

print("✅ 모든 라이브러리 import 완료!")

---
## STEP 1 ─ 예제 데이터 만들기

실제 Jupyter 노트북에서 파싱한 것처럼 **셀(cell) 목록**을 직접 만들어 봅니다.

각 셀은 딕셔너리(dict) 형태입니다:
```python
{
    'notebook':  'lecture1.ipynb',  # 어느 노트북의 셀인가?
    'cell_idx':  0,                  # 노트북 안에서 몇 번째 셀인가?
    'cell_type': 'code',             # 'code' 또는 'markdown'
    'source':    'x = 10\nprint(x)' # 실제 셀 내용
}
```

In [ ]:
# ── 예제: 두 개의 노트북(lecture1, lecture2)에서 추출한 셀 목록 ─────────────
#
# lecture1.ipynb: LangChain 기초 예제
# lecture2.ipynb: RAG 파이프라인 예제

cells = [
    # ── lecture1.ipynb 셀들 ────────────────────────────────────────────────
    {
        "notebook":  "lecture1.ipynb",
        "cell_idx":  0,
        "cell_type": "markdown",
        "source":    "# LangChain 기초\n이 노트북은 LangChain의 기본 사용법을 설명합니다."
    },
    {
        "notebook":  "lecture1.ipynb",
        "cell_idx":  1,
        "cell_type": "code",
        "source":    "from langchain_openai import ChatOpenAI\nmodel = ChatOpenAI(model='gpt-4o')"
        # 📌 'model' 변수 정의
    },
    {
        "notebook":  "lecture1.ipynb",
        "cell_idx":  2,
        "cell_type": "code",
        "source":    "from langchain_core.prompts import ChatPromptTemplate\nprompt = ChatPromptTemplate.from_messages([\n    ('system', '당신은 유용한 AI 어시스턴트입니다.'),\n    ('human', '{question}')\n])"
        # 📌 'prompt' 변수 정의
    },
    {
        "notebook":  "lecture1.ipynb",
        "cell_idx":  3,
        "cell_type": "code",
        "source":    "chain = prompt | model\nresult = chain.invoke({'question': 'LangChain이 뭐야?'})"
        # 📌 'chain', 'result' 변수 정의  ← prompt, model 재사용!
    },

    # ── lecture2.ipynb 셀들 ────────────────────────────────────────────────
    {
        "notebook":  "lecture2.ipynb",
        "cell_idx":  0,
        "cell_type": "markdown",
        "source":    "# RAG 파이프라인\n벡터 DB를 이용한 검색 증강 생성."
    },
    {
        "notebook":  "lecture2.ipynb",
        "cell_idx":  1,
        "cell_type": "code",
        "source":    "from langchain_openai import ChatOpenAI\nmodel = ChatOpenAI(model='gpt-4o-mini')"
        # 📌 'model' 변수 — lecture1과 **같은 변수명** 사용!
    },
    {
        "notebook":  "lecture2.ipynb",
        "cell_idx":  2,
        "cell_type": "code",
        "source":    "from langchain_community.vectorstores import FAISS\nvectorstore = FAISS.from_documents(docs, embeddings)\nretriever = vectorstore.as_retriever()"
        # 📌 'vectorstore', 'retriever' 변수 정의
    },
    {
        "notebook":  "lecture2.ipynb",
        "cell_idx":  3,
        "cell_type": "code",
        "source":    "chain = retriever | model\nresult = chain.invoke('RAG가 뭐야?')"
        # 📌 'chain', 'result' — lecture1과 **같은 변수명** 사용!
    },
]

print(f"총 {len(cells)}개의 셀 로드됨")
print(f"노트북 목록: {set(c['notebook'] for c in cells)}")
print()
print("첫 번째 셀 예시:")
pprint(cells[1])

---
## STEP 2 ─ `build_cell_graph` 함수 완전 해설

이 함수는 셀 목록을 받아 **방향 그래프(DiGraph)**를 만듭니다.

```
그래프 = 노드(Node) + 엣지(Edge)

노드  : 각각의 셀 1개
엣지  : 셀 간의 관계선
  ├─ sequential : 같은 노트북에서 순서가 인접한 셀 (cell0 → cell1)
  └─ shared_var : 두 셀에서 같은 변수명을 사용하는 경우
```

### 📊 완성된 그래프의 모습 (예상)

```
[lecture1#cell0] ──seq──▶ [lecture1#cell1] ──seq──▶ [lecture1#cell2] ──seq──▶ [lecture1#cell3]
                                │                                                      │
                            shared_var(model)                               shared_var(chain,result)
                                │                                                      │
[lecture2#cell0] ──seq──▶ [lecture2#cell1] ──seq──▶ [lecture2#cell2] ──seq──▶ [lecture2#cell3]
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 함수 정의 — 한 줄씩 상세 주석 포함
# ════════════════════════════════════════════════════════════════════════════

def build_cell_graph(cells: list[dict]) -> nx.DiGraph:
    """
    셀 목록을 입력받아 방향 그래프(DiGraph)를 반환합니다.

    Parameters
    ----------
    cells : list[dict]
        각 dict는 notebook, cell_idx, cell_type, source 키를 가집니다.

    Returns
    -------
    nx.DiGraph
        노드: 각 셀
        엣지:
          - sequential : 같은 노트북 내 순서
          - shared_var : 코드 셀 간 변수명 공유 (간단한 정규식)
    """

    # ── [블록 A] 빈 방향 그래프(DiGraph) 생성 ──────────────────────────────
    # DiGraph = Directed Graph = 방향이 있는 그래프
    # (A→B 와 B→A 는 서로 다른 엣지)
    G = nx.DiGraph()


    # ── [블록 B] 모든 셀을 노드로 추가 ───────────────────────────────────────
    for c in cells:

        # 노드 ID는 "노트북명#cell번호" 형식으로 고유하게 만들기
        # 예: "lecture1.ipynb#cell0", "lecture2.ipynb#cell3"
        node_id = f"{c['notebook']}#cell{c['cell_idx']}"

        # G.add_node(node_id, **kwargs):
        #   node_id  = 노드의 고유 이름
        #   source_text=c["source"] = 나중에 키워드 검색에 쓸 셀 내용을 별도 저장
        #   **c      = 딕셔너리의 나머지 키(notebook, cell_idx 등)도 속성으로 저장
        G.add_node(node_id, source_text=c["source"], **c)


    # ── [블록 C] sequential 엣지: 같은 노트북에서 인접한 셀끼리 연결 ──────────

    # C-1: 노트북별로 셀 그룹화
    #      nb_groups = { 'lecture1.ipynb': [cell0, cell1, ...],
    #                    'lecture2.ipynb': [cell0, cell1, ...] }
    nb_groups: dict[str, list[dict]] = {}
    for c in cells:
        # setdefault: 키가 없으면 빈 리스트로 초기화 후 append
        nb_groups.setdefault(c["notebook"], []).append(c)

    # C-2: 각 노트북별로 cell_idx 기준 정렬 후 인접 셀 연결
    for nb, nb_cells in nb_groups.items():

        # cell_idx 오름차순 정렬 (0, 1, 2, 3 ...)
        nb_cells_sorted = sorted(nb_cells, key=lambda x: x["cell_idx"])

        # 연속된 두 셀을 순서대로 엣지로 연결
        # i=0: cell0 → cell1
        # i=1: cell1 → cell2
        # i=2: cell2 → cell3
        for i in range(len(nb_cells_sorted) - 1):
            src = f"{nb}#cell{nb_cells_sorted[i]['cell_idx']}"
            tgt = f"{nb}#cell{nb_cells_sorted[i+1]['cell_idx']}"

            # rel="sequential" 속성을 붙여서 엣지 종류를 표시
            G.add_edge(src, tgt, rel="sequential")


    # ── [블록 D] shared_var 엣지: 변수명을 공유하는 코드 셀끼리 연결 ───────────

    # D-1: code 타입 셀만 추출 (markdown 셀은 변수 없음)
    code_cells = [c for c in cells if c["cell_type"] == "code"]

    # D-2: 변수 할당 패턴 정규식
    #   ^([a-zA-Z_]\w*)\s*=
    #   ^           : 줄의 시작
    #   [a-zA-Z_]   : 첫 글자는 영문자 또는 _
    #   \w*         : 이후 영문자, 숫자, _ 0회 이상
    #   \s*=        : 등호 앞에 공백 0개 이상
    #   re.MULTILINE: 소스 코드가 여러 줄이므로 ^ 를 각 줄 시작으로 처리
    #
    # 예) "model = ChatOpenAI(...)" → 'model' 추출
    #     "chain = prompt | model"  → 'chain' 추출
    assign_re = re.compile(r"^([a-zA-Z_]\w*)\s*=", re.MULTILINE)

    # D-3: 각 코드 셀에서 정의된 변수 집합 추출
    #      cell_vars = { 'lecture1.ipynb#cell1': {'model'},
    #                    'lecture1.ipynb#cell2': {'prompt'},
    #                    'lecture1.ipynb#cell3': {'chain', 'result'}, ... }
    cell_vars: dict[str, set[str]] = {}
    for c in code_cells:
        node_id = f"{c['notebook']}#cell{c['cell_idx']}"
        # findall은 정규식에서 ()그룹에 해당하는 문자열 리스트를 반환
        cell_vars[node_id] = set(assign_re.findall(c["source"]))

    # D-4: 모든 코드 셀 쌍(pair)을 비교해서 공통 변수가 있으면 엣지 추가
    node_ids = list(cell_vars.keys())

    for i, n1 in enumerate(node_ids):
        for j, n2 in enumerate(node_ids):

            # i >= j 이면 건너뜀 → 중복 방지 (n1-n2 쌍과 n2-n1 쌍을 동일 취급)
            if i >= j:
                continue

            # 교집합(&): 두 셀이 공통으로 사용하는 변수명
            shared = cell_vars[n1] & cell_vars[n2]

            if shared:
                # rel="shared_var" 속성으로 엣지 종류 표시
                # vars: 공유 변수명 목록 (최대 5개, 쉼표 구분)
                G.add_edge(n1, n2, rel="shared_var",
                           vars=",".join(list(shared)[:5]))

    return G


print("✅ build_cell_graph 함수 정의 완료")

---
## STEP 3 ─ 그래프 생성 및 구조 확인

In [ ]:
# 그래프 생성
G = build_cell_graph(cells)

# ── 기본 통계 출력 ─────────────────────────────────────────────────────────
print("=" * 50)
print("📊 그래프 기본 정보")
print("=" * 50)
print(f"노드(셀) 수   : {G.number_of_nodes()}")
print(f"엣지(관계) 수 : {G.number_of_edges()}")

# ── 모든 노드 출력 ─────────────────────────────────────────────────────────
print("\n📌 노드 목록:")
for node_id, data in G.nodes(data=True):
    print(f"  {node_id}")
    print(f"    cell_type : {data['cell_type']}")
    print(f"    source 첫줄: {data['source'].splitlines()[0][:60]}")

# ── 모든 엣지 출력 ─────────────────────────────────────────────────────────
print("\n🔗 엣지 목록:")
for src, tgt, data in G.edges(data=True):
    rel = data.get('rel', '?')
    extra = f" (공유 변수: {data['vars']})" if rel == 'shared_var' else ""
    print(f"  [{rel}] {src}  →  {tgt}{extra}")

---
## STEP 4 ─ 그래프 시각화 👁️

눈으로 확인하면 구조 이해가 훨씬 쉬워집니다!

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# ── 노드 위치 지정 (lecture1은 위, lecture2는 아래) ──────────────────────────
pos = {}
for node_id in G.nodes():
    nb, cell_part = node_id.split("#")
    cell_idx = int(cell_part.replace("cell", ""))
    x = cell_idx * 3.5          # 가로: cell 번호에 따라
    y = 1.0 if "lecture1" in nb else -1.0   # 세로: 노트북에 따라
    pos[node_id] = (x, y)

# ── 노드 색상 (코드셀=하늘색, 마크다운=연노랑) ─────────────────────────────
node_colors = []
for node_id in G.nodes():
    cell_type = G.nodes[node_id]['cell_type']
    node_colors.append("#AED6F1" if cell_type == "code" else "#FAD7A0")

# ── 엣지 분류 ────────────────────────────────────────────────────────────
seq_edges = [(s, t) for s, t, d in G.edges(data=True) if d.get('rel') == 'sequential']
var_edges = [(s, t) for s, t, d in G.edges(data=True) if d.get('rel') == 'shared_var']

# ── 그리기 ───────────────────────────────────────────────────────────────
# 노드
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2200, ax=ax)

# sequential 엣지 (회색, 실선)
nx.draw_networkx_edges(G, pos, edgelist=seq_edges,
                       edge_color="#5D6D7E", width=2, arrows=True,
                       arrowsize=20, connectionstyle="arc3,rad=0.05", ax=ax)

# shared_var 엣지 (빨강, 점선 효과를 위해 style='dashed')
nx.draw_networkx_edges(G, pos, edgelist=var_edges,
                       edge_color="#E74C3C", width=2, style="dashed", arrows=True,
                       arrowsize=20, connectionstyle="arc3,rad=0.3", ax=ax)

# 노드 레이블 (짧게)
labels = {n: n.split("#")[1] + "\n" + ("[code]" if G.nodes[n]['cell_type'] == 'code' else "[md]")
          for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=8, ax=ax)

# 엣지 레이블 (shared_var만)
edge_labels = {(s, t): d['vars'] for s, t, d in G.edges(data=True) if d.get('rel') == 'shared_var'}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=7, font_color="#C0392B", ax=ax)

# ── 범례 및 제목 ─────────────────────────────────────────────────────────
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#5D6D7E', linewidth=2, label='sequential (순서 연결)'),
    Line2D([0], [0], color='#E74C3C', linewidth=2, linestyle='dashed', label='shared_var (변수 공유)'),
    plt.Rectangle((0, 0), 1, 1, fc='#AED6F1', label='code cell'),
    plt.Rectangle((0, 0), 1, 1, fc='#FAD7A0', label='markdown cell'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

# 노트북 레이블
ax.text(-0.5, 1.0, "lecture1.ipynb", fontsize=11, fontweight='bold', color='#2980B9',
        va='center')
ax.text(-0.5, -1.0, "lecture2.ipynb", fontsize=11, fontweight='bold', color='#27AE60',
        va='center')

ax.set_title("Cell Graph 시각화", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("  회색 실선 → sequential: 같은 노트북 안에서 순서대로 연결")
print("  빨간 점선 → shared_var: 두 노트북 간에 같은 변수명(model, chain, result) 공유")

---
## STEP 5 ─ 변수 추출 동작 확인 (정규식 이해)

`shared_var` 엣지가 어떻게 만들어지는지 정규식 동작을 직접 확인합니다.

In [ ]:
# 정규식: 변수 할당 패턴 추출
assign_re = re.compile(r"^([a-zA-Z_]\w*)\s*=", re.MULTILINE)

print("=" * 55)
print("🔬 각 코드 셀에서 추출된 변수명")
print("=" * 55)

for c in cells:
    if c['cell_type'] != 'code':
        continue
    node_id = f"{c['notebook']}#cell{c['cell_idx']}"
    found_vars = set(assign_re.findall(c['source']))
    print(f"\n📄 {node_id}")
    print(f"   소스 코드:")
    for line in c['source'].splitlines():
        print(f"     {line}")
    print(f"   추출된 변수: {found_vars}")

print()
print("=" * 55)
print("🔍 변수 공유 분석")
print("=" * 55)

# 실제 그래프에서 shared_var 엣지 분석
for src, tgt, data in G.edges(data=True):
    if data.get('rel') == 'shared_var':
        print(f"\n  {src}")
        print(f"  ↕ 공유 변수: [{data['vars']}]")
        print(f"  {tgt}")

---
## STEP 6 ─ `graph_search` 함수 완전 해설

이 함수가 하는 일을 큰 그림으로 보면:

```
입력: 사용자 질문 (query)

  ┌─────────────────────────────────────────────────┐
  │  Step 1: Seed 노드 선정                           │
  │    1a. 벡터 유사도 검색으로 질문과 비슷한 셀 찾기    │
  │    1b. 키워드 매칭으로 점수 보완                    │
  └──────────────────────┬──────────────────────────┘
                         ↓
  ┌─────────────────────────────────────────────────┐
  │  Step 2: 점수 전파 (Multi-hop)                   │
  │    seed 노드에서 이웃 노드로 점수 퍼뜨리기           │
  │    shared_var 이웃: 80% 점수 전달 (높은 관련성)     │
  │    sequential 이웃: 50% 점수 전달 (낮은 관련성)     │
  └──────────────────────┬──────────────────────────┘
                         ↓
  ┌─────────────────────────────────────────────────┐
  │  Step 3: 상위 top_k 반환                          │
  └─────────────────────────────────────────────────┘

출력: 관련도 높은 셀 문서(Document) 리스트
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# graph_search 함수 — 한 줄씩 상세 주석 포함
# ════════════════════════════════════════════════════════════════════════════

def graph_search(
    G: nx.DiGraph,
    docs: list[Document],
    query: str,
    vector_retriever,
    top_k: int = 5,
    hops: int = 2,
    seq_decay: float = 0.5,
    var_decay: float = 0.8
) -> list[Document]:
    """
    개선된 Graph RAG 검색 함수.

    Parameters
    ----------
    G              : build_cell_graph로 만든 방향 그래프
    docs           : 검색 가능한 Document 객체 목록
                     (각 doc.metadata['source'] = 노드 ID)
    query          : 사용자 질문 문자열
    vector_retriever: 벡터 유사도 검색 객체 (invoke 메서드 필요)
    top_k          : 최종 반환할 문서 개수 (기본 5)
    hops           : 그래프 탐색 깊이 (기본 2)
                     hops=1: seed의 이웃까지 탐색
                     hops=2: seed의 이웃의 이웃까지 탐색
    seq_decay      : sequential 엣지 점수 감쇠율 (기본 0.5)
    var_decay      : shared_var 엣지 점수 감쇠율 (기본 0.8)
    """

    # ── 도우미: node_id → Document 매핑 사전 ────────────────────────────────
    # 나중에 node_id로 Document를 빠르게 찾기 위해 사전으로 변환
    # doc.metadata['source'] 가 노드 ID (예: 'lecture1.ipynb#cell1')
    doc_map = {d.metadata["source"]: d for d in docs}


    # ══════════════════════════════════════════════════════════════════════
    # Step 1a: 벡터 유사도로 Seed 노드 선정
    # ══════════════════════════════════════════════════════════════════════
    #
    # vector_retriever.invoke(query) 는 내부적으로:
    #   1) query 를 임베딩 벡터로 변환
    #   2) 벡터 DB에서 코사인 유사도가 높은 문서 반환
    #
    # 반환된 상위 3개 문서를 초기 점수 1.0 으로 설정
    try:
        seed_docs = vector_retriever.invoke(query)
        scores: dict[str, float] = {
            d.metadata["source"]: 1.0      # 초기 점수 = 1.0 (최고점)
            for d in seed_docs[:3]          # 상위 3개만
            if d.metadata["source"] in G    # 그래프에 있는 노드만 포함
        }
    except Exception:
        # 벡터 검색 실패 시 빈 dict로 시작 (키워드 검색으로만 진행)
        scores = {}


    # ══════════════════════════════════════════════════════════════════════
    # Step 1b: [Option 1] 키워드 매칭으로 보조 점수 부여
    # ══════════════════════════════════════════════════════════════════════
    #
    # 벡터 검색은 의미 유사도를 보지만 특정 키워드를 놓칠 수 있음
    # 예) 'ChatOpenAI' → 벡터 검색으로 못 찾을 수도 있음
    # 키워드 매칭으로 보완!

    query_lower = query.lower()   # 대소문자 구분 없이 비교

    # 불용어(stopwords): 검색에 의미 없는 단어들 제거
    # 영어 불용어 + 한국어 조사 포함
    stopwords = {"", "the", "a", "is", "in", "of", "for", "and", "or", "to",
                 "이", "가", "를", "은", "는", "의", "에", "도"}

    # 쿼리를 토큰(단어)으로 분리 (비단어 문자 기준으로 split)
    # 예) "ChatOpenAI model은 어떻게?" → {'chatopenai', 'model', '어떻게'}
    tokens = set(re.split(r"\W+", query_lower)) - stopwords

    if tokens:
        for node_id, data in G.nodes(data=True):
            # 노드에 저장된 실제 셀 내용 (build_cell_graph에서 source_text로 저장)
            cell_text = data.get("source_text", "").lower()

            # 이 셀에 쿼리 토큰이 몇 개나 포함되는가?
            kw_score = sum(1 for t in tokens if t in cell_text)

            if kw_score > 0:
                # 키워드 점수를 0 ~ 0.4 범위로 정규화
                # (벡터 점수 1.0보다 낮게 → 어디까지나 보조)
                # 예) 토큰 3개 중 2개 매칭: (2/3)*0.4 ≈ 0.27
                boost = min(kw_score / len(tokens), 1.0) * 0.4

                # max()로 기존 벡터 점수보다 낮으면 덮어쓰지 않음
                scores[node_id] = max(scores.get(node_id, 0), boost)

    # Seed 점수가 하나도 없으면 빈 리스트 반환 (관련 셀 없음)
    if not scores:
        return []


    # ══════════════════════════════════════════════════════════════════════
    # Step 2: [Option 3] 가중치 기반 Multi-hop 점수 전파
    # ══════════════════════════════════════════════════════════════════════
    #
    # 아이디어:
    #   seed 노드가 점수 1.0을 가지고 있으면,
    #   그 이웃 노드에 점수를 감쇠해서 전달
    #
    # shared_var 이웃: 0.8 감쇠  (변수 공유 → 강하게 연관)
    # sequential 이웃: 0.5 감쇠  (순서 인접 → 약하게 연관)
    #
    # hops=2 이면 이 과정을 2번 반복:
    #   1번째: seed(1.0) → 이웃1(0.8 or 0.5)
    #   2번째: 이웃1(0.8) → 이웃2(0.64 or 0.4)

    for hop_num in range(hops):  # hops 횟수만큼 반복 (기본 2번)

        # 현재 scores를 복사해서 수정 (원본 변경 방지)
        new_scores = dict(scores)

        for node, score in scores.items():

            # 이 노드가 그래프에 없으면 건너뜀
            if node not in G:
                continue

            # ── 나가는 엣지(outgoing): node → neighbor ──────────────────
            for neighbor in G.successors(node):
                # 이 엣지의 rel 속성 확인 (sequential? shared_var?)
                rel = G.get_edge_data(node, neighbor, default={}).get("rel", "")
                weight = var_decay if rel == "shared_var" else seq_decay

                # max()로 이미 높은 점수가 있으면 덮어쓰지 않음
                new_scores[neighbor] = max(
                    new_scores.get(neighbor, 0.0),
                    score * weight
                )

            # ── 들어오는 엣지(incoming): neighbor → node ──────────────────
            # 방향 반대도 탐색! (양방향 전파)
            for neighbor in G.predecessors(node):
                rel = G.get_edge_data(neighbor, node, default={}).get("rel", "")
                weight = var_decay if rel == "shared_var" else seq_decay

                new_scores[neighbor] = max(
                    new_scores.get(neighbor, 0.0),
                    score * weight
                )

        # 이번 hop 결과를 다음 hop의 입력으로
        scores = new_scores


    # ══════════════════════════════════════════════════════════════════════
    # Step 3: 점수 기준 내림차순 정렬 후 상위 top_k 반환
    # ══════════════════════════════════════════════════════════════════════

    # key=lambda x: -x[1]  →  점수(x[1])를 음수로 → 높은 점수가 앞으로
    ranked = sorted(scores.items(), key=lambda x: -x[1])

    # doc_map에 있는 노드만 반환 (Document 객체가 있는 경우만)
    return [doc_map[nid] for nid, _ in ranked[:top_k] if nid in doc_map]


print("✅ graph_search 함수 정의 완료")

---
## STEP 7 ─ 점수 전파 시뮬레이션 (핵심 알고리즘 이해)

`graph_search`의 가장 중요한 부분인 **점수 전파(score propagation)**를 직접 시뮬레이션합니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 점수 전파 시뮬레이션
# 시나리오: 'lecture1.ipynb#cell1' (model 정의 셀)이 seed로 선정됨
# ══════════════════════════════════════════════════════════════════════

# 초기 seed 점수 설정 (벡터 검색 결과라고 가정)
scores = {"lecture1.ipynb#cell1": 1.0}
seq_decay = 0.5
var_decay = 0.8

print("=" * 60)
print("📡 점수 전파 시뮬레이션")
print("=" * 60)
print(f"\n초기 seed: lecture1.ipynb#cell1 = 1.0")
print(f"  (이 셀에서 'model' 변수가 정의됨)")

for hop in range(2):  # 2번 반복
    print(f"\n{'─'*50}")
    print(f"🔄 Hop {hop+1} 시작")
    print(f"{'─'*50}")
    new_scores = dict(scores)

    for node, score in scores.items():
        if node not in G:
            continue

        print(f"\n  처리 중: {node} (점수={score:.3f})")

        # 나가는 엣지
        for neighbor in G.successors(node):
            rel = G.get_edge_data(node, neighbor, default={}).get("rel", "")
            weight = var_decay if rel == "shared_var" else seq_decay
            new_score = score * weight
            old = new_scores.get(neighbor, 0.0)
            new_scores[neighbor] = max(old, new_score)
            print(f"    → [{rel}] {neighbor}")
            print(f"       점수: {score:.3f} × {weight} = {new_score:.3f}  "
                  f"(기존: {old:.3f} → 최종: {new_scores[neighbor]:.3f})")

        # 들어오는 엣지
        for neighbor in G.predecessors(node):
            rel = G.get_edge_data(neighbor, node, default={}).get("rel", "")
            weight = var_decay if rel == "shared_var" else seq_decay
            new_score = score * weight
            old = new_scores.get(neighbor, 0.0)
            new_scores[neighbor] = max(old, new_score)
            print(f"    ← [{rel}] {neighbor}")
            print(f"       점수: {score:.3f} × {weight} = {new_score:.3f}  "
                  f"(기존: {old:.3f} → 최종: {new_scores[neighbor]:.3f})")

    scores = new_scores

print(f"\n{'='*60}")
print("📊 최종 점수 결과")
print(f"{'='*60}")
for node_id, score in sorted(scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 20)
    print(f"  {score:.3f} {bar:20s} {node_id}")

---
## STEP 8 ─ 전체 파이프라인 실행

실제로 `graph_search`를 사용하려면 벡터 검색기(retriever)가 필요합니다.
여기서는 **Mock Retriever**를 만들어서 전체 흐름을 테스트합니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Document 객체 생성
# 각 셀을 LangChain Document로 변환
# metadata['source'] = 노드 ID (그래프와 연결 고리!)
# ══════════════════════════════════════════════════════════════════════

docs = [
    Document(
        page_content=c["source"],
        metadata={
            "source": f"{c['notebook']}#cell{c['cell_idx']}",
            "notebook": c["notebook"],
            "cell_idx": c["cell_idx"],
            "cell_type": c["cell_type"],
        }
    )
    for c in cells
]

print(f"총 {len(docs)}개의 Document 생성")
print("\n첫 번째 Document:")
print(f"  page_content: {docs[1].page_content[:60]}...")
print(f"  metadata    : {docs[1].metadata}")


# ══════════════════════════════════════════════════════════════════════
# Mock Vector Retriever (실제 임베딩 대신 사용)
# 실제로는 FAISS, Chroma 등의 vectorstore.as_retriever() 사용
# ══════════════════════════════════════════════════════════════════════

class MockVectorRetriever:
    """
    테스트용 가짜 벡터 검색기.
    실제 구현에서는 아래처럼 사용합니다:
    
        from langchain_community.vectorstores import FAISS
        from langchain_openai import OpenAIEmbeddings
        
        vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())
        retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
    """
    def __init__(self, seed_node_ids: list[str], all_docs: list[Document]):
        # seed_node_ids: 벡터 검색에서 반환할 노드 ID 목록 (테스트용)
        self.seed_node_ids = seed_node_ids
        self.doc_map = {d.metadata['source']: d for d in all_docs}

    def invoke(self, query: str) -> list[Document]:
        # 실제로는 query를 임베딩해서 유사한 문서를 반환
        # 여기서는 미리 지정한 노드 ID의 문서를 반환
        return [
            self.doc_map[nid]
            for nid in self.seed_node_ids
            if nid in self.doc_map
        ]


# 시나리오: 'ChatOpenAI model 초기화' 쿼리 시
#   → 벡터 검색으로 lecture1#cell1 이 반환된다고 가정
mock_retriever = MockVectorRetriever(
    seed_node_ids=["lecture1.ipynb#cell1"],
    all_docs=docs
)

print("\n✅ Mock Retriever 생성 완료")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# graph_search 실행!
# ══════════════════════════════════════════════════════════════════════

query = "ChatOpenAI model 초기화"

print("=" * 60)
print(f"🔍 쿼리: '{query}'")
print("=" * 60)

result_docs = graph_search(
    G=G,
    docs=docs,
    query=query,
    vector_retriever=mock_retriever,
    top_k=5,
    hops=2,
    seq_decay=0.5,
    var_decay=0.8
)

print(f"\n📋 검색 결과 ({len(result_docs)}개):\n")
for i, doc in enumerate(result_docs, 1):
    print(f"  [{i}] {doc.metadata['source']}")
    print(f"      타입: {doc.metadata['cell_type']}")
    print(f"      내용: {doc.page_content[:80].replace(chr(10), ' ')}")
    print()

print("💡 단순 벡터 검색이라면 cell1만 반환했을 것입니다.")
print("   Graph RAG는 연결된 셀들(shared_var/sequential)도 함께 가져옵니다!")

---
## STEP 9 ─ 파라미터 실험: `hops`와 `decay`가 결과에 미치는 영향

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 점수 전파를 직접 계산해서 파라미터 효과 시각화
# ══════════════════════════════════════════════════════════════════════

def compute_scores_only(G, seed_node, hops, seq_decay, var_decay):
    """점수 전파만 수행하는 헬퍼 함수 (그래프 탐색 순수 추출)"""
    scores = {seed_node: 1.0}
    for _ in range(hops):
        new_scores = dict(scores)
        for node, score in scores.items():
            if node not in G:
                continue
            for neighbor in G.successors(node):
                rel = G.get_edge_data(node, neighbor, default={}).get("rel", "")
                w = var_decay if rel == "shared_var" else seq_decay
                new_scores[neighbor] = max(new_scores.get(neighbor, 0.0), score * w)
            for neighbor in G.predecessors(node):
                rel = G.get_edge_data(neighbor, node, default={}).get("rel", "")
                w = var_decay if rel == "shared_var" else seq_decay
                new_scores[neighbor] = max(new_scores.get(neighbor, 0.0), score * w)
        scores = new_scores
    return scores


seed = "lecture1.ipynb#cell1"
node_list = sorted(G.nodes())
configs = [
    ("hops=1, decay 기본",   1, 0.5, 0.8),
    ("hops=2, decay 기본",   2, 0.5, 0.8),
    ("hops=2, seq_decay=0.9", 2, 0.9, 0.8),
    ("hops=2, var_decay=0.3", 2, 0.5, 0.3),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

short_labels = [n.split("#")[1] + "\n" + n.split("#")[0].replace(".ipynb", "") for n in node_list]

for ax, (title, hops, sd, vd) in zip(axes, configs):
    sc = compute_scores_only(G, seed, hops, sd, vd)
    values = [sc.get(n, 0.0) for n in node_list]
    colors = ["#E74C3C" if n == seed else "#3498DB" for n in node_list]
    bars = ax.bar(short_labels, values, color=colors)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.tick_params(axis='x', labelsize=7)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                    f"{val:.2f}", ha='center', va='bottom', fontsize=8)

fig.suptitle(f"파라미터별 점수 전파 비교\n(빨강: seed '{seed}')",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 관찰 포인트:")
print("  • hops를 늘리면 더 멀리 있는 셀까지 점수가 전파됩니다.")
print("  • seq_decay를 높이면 순서 인접 셀이 더 많은 점수를 받습니다.")
print("  • var_decay를 낮추면 변수 공유 셀의 이점이 사라집니다.")

---
## STEP 10 ─ 실제 OpenAI API 연동 (선택사항)

실제 임베딩과 벡터 검색기를 사용하는 전체 파이프라인입니다.
OpenAI API 키가 있다면 아래 코드를 실행해보세요.

In [ ]:
# ⚠️ 이 셀은 OPENAI_API_KEY 환경변수가 필요합니다!
# 실행 전에 아래 주석을 해제하거나 환경변수를 설정하세요.

# import os
# os.environ["OPENAI_API_KEY"] = "sk-..."

OPENAI_AVAILABLE = False  # API 키가 있으면 True로 변경

if OPENAI_AVAILABLE:
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS

    # 1. 임베딩 모델 생성
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    # 2. FAISS 벡터 스토어 생성 (docs를 임베딩해서 저장)
    vectorstore = FAISS.from_documents(docs, embeddings)

    # 3. Retriever 생성
    real_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 4. graph_search 실행
    query = "ChatOpenAI model을 어떻게 초기화하나요?"
    results = graph_search(
        G=G,
        docs=docs,
        query=query,
        vector_retriever=real_retriever,
        top_k=5,
        hops=2
    )

    print(f"쿼리: {query}")
    print(f"결과 {len(results)}개:")
    for r in results:
        print(f"  - {r.metadata['source']}")
else:
    print("ℹ️ OPENAI_AVAILABLE = False")
    print("   OpenAI API 키를 설정하고 OPENAI_AVAILABLE = True로 변경하면")
    print("   실제 임베딩을 사용한 Graph RAG를 실행할 수 있습니다.")

---
## 📝 전체 요약

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Graph RAG 파이프라인 요약                           │
├──────────────────────┬──────────────────────────────────────────────┤
│  build_cell_graph()  │  셀 목록 → NetworkX 방향 그래프                │
│                      │  ├─ 노드: 각 셀 (노트북명#cell번호)            │
│                      │  ├─ 엣지 sequential: 순서 인접 셀              │
│                      │  └─ 엣지 shared_var: 변수명 공유 셀            │
├──────────────────────┼──────────────────────────────────────────────┤
│  graph_search()      │  질문 → 관련 Document 리스트                   │
│                      │  ├─ Step1a: 벡터 유사도 → seed 노드 (점수=1.0) │
│                      │  ├─ Step1b: 키워드 매칭 → 보조 점수 (+0~0.4)   │
│                      │  ├─ Step2:  Multi-hop 점수 전파 (감쇠)         │
│                      │  │    shared_var: ×0.8  sequential: ×0.5      │
│                      │  └─ Step3: 상위 top_k 문서 반환                │
├──────────────────────┴──────────────────────────────────────────────┤
│  왜 Graph RAG인가?                                                    │
│    단순 벡터 검색: 의미론적으로 가까운 셀 1~3개만 반환                   │
│    Graph RAG   : 관련 셀 + 그것과 연결된(의존하는) 셀까지 함께 반환     │
│    → LLM이 코드 맥락을 훨씬 잘 이해할 수 있음!                         │
└─────────────────────────────────────────────────────────────────────┘
```

### 주요 파라미터 치트시트

| 파라미터 | 기본값 | 역할 | 올리면 | 낮추면 |
|---------|--------|------|--------|--------|
| `hops` | 2 | 탐색 깊이 | 더 먼 셀까지 탐색 | 직접 연결된 셀만 |
| `seq_decay` | 0.5 | 순서 엣지 감쇠 | 순서 이웃 더 중요 | 순서 이웃 덜 중요 |
| `var_decay` | 0.8 | 변수 엣지 감쇠 | 변수 공유 더 중요 | 변수 공유 덜 중요 |
| `top_k` | 5 | 반환 문서 수 | 더 많은 문서 → LLM 컨텍스트 길어짐 | 적은 문서 |
